In [5]:
import pandas as pd
import ase
import os
import copy
import numpy as np
from pprint import pprint
from ase.io.vasp import read_vasp, write_vasp
from ase.visualize import view

In [2]:
pt_2o_df = pd.read_csv('/home/jgeiger/WAREHOUSE/projects/3_ceria_invite/calcs/pt/2o_coord/default/analysis/pt_2o_default_final.csv', sep=';')
pt_2o_df.shape
pt_2o_df = pt_2o_df.sort_values(by=['mos', 'ce3_ids'])
pt_2o_df['dir'] = pt_2o_df['dir'].str.replace('3_ceria_au', '3_ceria_invite')
ase_list = [read_vasp(os.path.join(_, 'CONTCAR')) for _ in pt_2o_df['dir']]
pt_2o_df['ase_contcar'] = ase_list

(11, 10)

In [3]:
pt_2o_df.head()

,dir,metal,ener,ce3_ids,ncoord,mos_init,mos,mgn_cont,E_rel_global,uniq_ce3_ids,ase_contcar
7,/media/jgeiger/WAREHOUSE/projects/3_ceria_invi...,Pt,-856.948905,0,2,0,0,0.000,0.278669,0,"(Atom('Ce', [1.8849538053882702, 3.84391568438..."
4,/media/jgeiger/WAREHOUSE/projects/3_ceria_invi...,Pt,-857.067723,30,2,1,1,1.348,0.159851,30,"(Atom('Ce', [1.8849538053882702, 3.84391568438..."
3,/media/jgeiger/WAREHOUSE/projects/3_ceria_invi...,Pt,-857.125837,31,2,1,1,1.349,0.101737,31,"(Atom('Ce', [1.8849538053882702, 3.84391568438..."
2,/media/jgeiger/WAREHOUSE/projects/3_ceria_invi...,Pt,-857.128586,32,2,1,1,1.351,0.098988,32,"(Atom('Ce', [1.8849538053882702, 3.84391568438..."
5,/media/jgeiger/WAREHOUSE/projects/3_ceria_invi...,Pt,-857.062277,33,2,1,1,1.349,0.165297,33,"(Atom('Ce', [1.8849538053882702, 3.84391568438..."


In [8]:
from fileinput import filename


devel_ase = pt_2o_df['ase_contcar'][0]
fixed_symbols = devel_ase.get_chemical_symbols()
ce3_ids_list = pt_2o_df['ce3_ids'].values
ce3_ids_list = [list(map(int, _.split('_'))) for _ in ce3_ids_list]
mos_list = [int(_) for _ in pt_2o_df['mos'].values]
mos_element_list = ['Fm', 'Md', 'No']

# mos_dict =

ase_mod_list = []

for iase_structure, ase_structure in enumerate(ase_list):
    mod_symbols = [] # copy.deepcopy(fixed_symbols)

    print(iase_structure+1, ce3_ids_list[iase_structure])
    for isymbol, symbol in enumerate(fixed_symbols):

        if isymbol+1 in ce3_ids_list[iase_structure]:
            mod_symbols.append('La')
        else:
            mod_symbols.append(symbol)

    ase_mod_structure = copy.deepcopy(ase_structure)
    ase_mod_structure.set_chemical_symbols(mod_symbols)
    # pt_position = np.array(list([atom for atom in ase_mod_structure if atom.symbol=='Pt'][0].position[:2])+[0])
    pt_position = np.array(list([atom for atom in ase_mod_structure if atom.index==0][0].position[:2])+[0])
    cell_center = np.array(list(ase_mod_structure.get_cell().lengths()/2)[:2]+[0])

    ase_mod_structure.translate(cell_center-pt_position)
    ase_mod_structure.wrap()

    ase_mod_structure[-1].symbol = mos_element_list[mos_list[iase_structure]]

    # ase_mod_structure = ase_mod_structure.repeat([3,3,1])

    ase_mod_list.append(ase_mod_structure)
    write_vasp(
        atoms=ase_mod_structure,
        file='/home/jgeiger/Documents/iciq/phd_day_2022/videos/pt_2o_polaron_dists/pt_2o_{}_{}.vasp'.format(mos_list[iase_structure], iase_structure)
        )
        # images=ase_mod_list, filename=os.path.join('/home/jgeiger/Documents/iciq/phd_day_2022/videos/pt_2o_polaron_dists/pt_2o_polaron_dists.xyz'), format='extxyz'
        
#     view(ase_mod_structure)
#     break
# view(ase_mod_list)
# ase.io.write(images=ase_mod_list, filename=os.path.join('/home/jgeiger/Documents/iciq/phd_day_2022/videos/pt_2o_polaron_dists/pt_2o_polaron_dists.xyz'), format='extxyz')

1 [0]
2 [30]
3 [31]
4 [32]
5 [33]
6 [35]
7 [28, 29]
8 [29, 31]
9 [30, 34]
10 [31, 32]
11 [33, 34]


In [6]:
pprint(ce3_ids_list)


[[35],
 [34],
 [29],
 [28],
 [28, 29],
 [32],
 [31],
 [30],
 [36],
 [33],
 [29, 31],
 [0],
 [33, 34],
 [30, 34],
 [31, 32]]
